In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import (skew, 
                        kurtosis)
from scipy.signal import welch

# Extract Time-Related Features

In [22]:
def extract_time_features(X):
    """Extract time-domain features"""
    n_channels, n_samples = X.shape
    features = []
    
    for ch in range(n_channels):
        signal = X[ch, :]
        
        # Statistical features
        trial_features = [
            np.mean(signal),          # Mean
            np.std(signal),           # Standard deviation
            np.var(signal),           # Variance
            skew(signal),             # Skewness
            kurtosis(signal),         # Kurtosis
            np.max(signal),           # Maximum
            np.min(signal),           # Minimum
            np.ptp(signal),           # Peak-to-peak
            np.median(signal),        # Median
            np.mean(np.abs(signal)),  # Mean absolute value
            np.sum(np.diff(np.sign(signal)) != 0),  # Zero crossing rate
            np.sqrt(np.mean(signal**2)),            # Root mean square
        ]
        
        features.extend(trial_features)
    
    return pd.Series(features)

In [23]:
X_train_features = data["epoch_data_ICA"].apply(extract_time_features)

# Optional: give columns meaningful names
n_channels = 64
feature_names = ['mean', 'std', 'var', 'skew', 'kurt', 'max', 'min', 'ptp', 'median', 'mean_abs', 'zero_cross', 'rms']
columns = [f"{ch+1}_{feat}" for ch in range(n_channels) for feat in feature_names]
X_train_features.columns = columns

# Now X_train_features is a DataFrame ready for ML
print(X_train_features.shape)  # (n_epochs, n_channels*12)

KeyboardInterrupt: 

In [ ]:
X_train_features

In [3]:
def extract_frequency_features_trial(signal, sfreq, freq_bands):
    """
    Extract frequency-domain features for a single EEG trial (n_channels x n_samples)
    """
    n_channels, n_samples = signal.shape
    trial_features = []

    for ch in range(n_channels):
        sig = signal[ch, :]
        
        # Fill NaNs just in case
        sig = np.nan_to_num(sig, 0.0)
        
        # Power spectral density
        freqs, psd = welch(sig, fs=sfreq, nperseg=min(256, n_samples//4))
        
        # Band power features
        for band_name, (low, high) in freq_bands.items():
            band_mask = (freqs >= low) & (freqs <= high)
            band_power = np.sum(psd[band_mask])
            trial_features.append(band_power)
        
        # Spectral features
        total_power = np.sum(psd)
        spectral_centroid = np.sum(freqs * psd) / total_power
        spectral_spread = np.sqrt(np.sum(((freqs - spectral_centroid)**2) * psd) / total_power)
        spectral_entropy = -np.sum((psd / total_power) * np.log2(psd / total_power + 1e-12))
        
        trial_features.extend([
            total_power,
            spectral_centroid,
            spectral_spread,
            spectral_entropy
        ])
        
        # Peak frequency
        peak_freq = freqs[np.argmax(psd)]
        trial_features.append(peak_freq)
    
    return pd.Series(trial_features)

In [10]:
data[(data["epoch_data_ICA"]=="Error")]

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [16]:
data["factor"]=data["epoch_data_ICA"].apply(lambda  x : "str" if type(x) == str else "not_str")

In [24]:
data.shape

(27547, 9)